# Notebook 02 — Topology Drift Demo

**Repo:** `residual-phase-lock`  
**Notebook:** `02_topology_drift_demo.ipynb`

## Claim

> Models can achieve local fit while drifting from global structure.

Notebook 01 showed:

```text
residual ≠ noise
residual → structure signal
```

Notebook 02 shows the next step:

```text
accuracy ≠ coherence
local fit → topology drift
```

We use a parity task because parity is a clean global constraint: the correct label depends on the whole binary vector, not on any single local feature.

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f"""
## Interpretation

```text
{interpretation.strip()}
```
"""
            md = f"""# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
"""
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(43)
NOTEBOOK_ID = "02"
NOTEBOOK_SLUG = "topology_drift_demo"
exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Define a global-structure task

We generate binary vectors and label each vector by parity:

```text
label = sum(bits) mod 2
```

Parity is useful here because the label is a global property of the full vector.

In [ ]:
def make_parity_data(n_samples=5000, dim=16, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 2, size=(n_samples, dim))
    y = X.sum(axis=1) % 2
    return X, y

DIM = 16
X_train, y_train = make_parity_data(n_samples=2500, dim=DIM, seed=1)
X_test, y_test = make_parity_data(n_samples=5000, dim=DIM, seed=2)

train_df = pd.DataFrame(X_train, columns=[f"bit_{i}" for i in range(DIM)])
train_df["parity"] = y_train
test_df = pd.DataFrame(X_test, columns=[f"bit_{i}" for i in range(DIM)])
test_df["parity"] = y_test

exp.save_csv(train_df, "train_parity_data")
exp.save_csv(test_df, "test_parity_data")

train_df.head()

## 3. Train a local-fit model

We use a simple MLP classifier. This is not meant to be a state-of-the-art parity solver. It is intentionally a minimal model that can fit local patterns while still drifting from the global parity constraint.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=43,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_accuracy = float(accuracy_score(y_train, y_train_pred))
test_accuracy = float(accuracy_score(y_test, y_test_pred))

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")

## 4. Measure topology drift

For this controlled task, the global rule is known. Drift occurs whenever the model output violates the parity rule.

```text
drift = predicted_parity ≠ true_parity
```

This is a discrete version of structure drift: the model output leaves the global constraint manifold.

In [ ]:
train_drift = (y_train_pred != y_train).astype(int)
test_drift = (y_test_pred != y_test).astype(int)

train_drift_rate = float(train_drift.mean())
test_drift_rate = float(test_drift.mean())

train_residual = y_train - y_train_pred
test_residual = y_test - y_test_pred

print(f"Train drift rate: {train_drift_rate:.4f}")
print(f"Test drift rate:  {test_drift_rate:.4f}")

drift_df = pd.DataFrame({
    "true_parity": y_test,
    "predicted_parity": y_test_pred,
    "residual": test_residual,
    "drift": test_drift,
    "hamming_weight": X_test.sum(axis=1),
})

exp.save_csv(drift_df, "test_drift_table")
drift_df.head()

## 5. Accuracy vs drift

Accuracy and drift describe the same binary outcomes from opposite directions, but drift is the more useful framing for this repo:

```text
accuracy = local performance summary
drift = structural violation signal
```

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "train_accuracy",
        "test_accuracy",
        "train_drift_rate",
        "test_drift_rate",
        "generalization_gap",
    ],
    "value": [
        train_accuracy,
        test_accuracy,
        train_drift_rate,
        test_drift_rate,
        train_accuracy - test_accuracy,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")
summary

In [ ]:
plt.figure(figsize=(7, 4))
labels = ["Train accuracy", "Test accuracy", "Train drift", "Test drift"]
values = [train_accuracy, test_accuracy, train_drift_rate, test_drift_rate]
plt.bar(labels, values)
plt.ylim(0, 1)
plt.ylabel("rate")
plt.title("Accuracy and topology drift")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
exp.save_fig("accuracy_vs_drift")
plt.show()

## 6. Drift by Hamming weight

Parity depends on whether the Hamming weight is even or odd. Grouping drift by Hamming weight checks whether errors are structure-dependent rather than featureless noise.

In [ ]:
drift_by_weight = (
    drift_df.groupby("hamming_weight")
    .agg(
        count=("drift", "size"),
        drift_rate=("drift", "mean"),
        mean_residual=("residual", "mean"),
    )
    .reset_index()
)

exp.save_csv(drift_by_weight, "drift_by_hamming_weight")
drift_by_weight

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(drift_by_weight["hamming_weight"], drift_by_weight["drift_rate"], marker="o")
plt.ylim(0, 1)
plt.xlabel("Hamming weight")
plt.ylabel("drift rate")
plt.title("Topology drift by Hamming weight")
plt.grid(True, alpha=0.3)
plt.tight_layout()
exp.save_fig("drift_by_hamming_weight")
plt.show()

## 7. Residual distribution

The residual encodes the direction of parity violation:

```text
+1 → predicted 0 when true label is 1
-1 → predicted 1 when true label is 0
0  → no drift
```

Notebook 01 showed that residuals can carry structure. Notebook 02 shows residuals as direct topology-drift signals.

In [ ]:
residual_counts = (
    pd.Series(test_residual)
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="count")
)

exp.save_csv(residual_counts, "residual_counts")

plt.figure(figsize=(6, 4))
plt.bar(residual_counts["residual"].astype(str), residual_counts["count"])
plt.xlabel("residual")
plt.ylabel("count")
plt.title("Residual distribution for topology drift")
plt.tight_layout()
exp.save_fig("residual_distribution")
plt.show()

residual_counts

## 8. Confusion matrix

The confusion matrix shows how the classifier drifts between the two parity states.

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"])
exp.save_csv(cm_df.reset_index().rename(columns={"index": "label"}), "confusion_matrix")

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Parity confusion matrix")
plt.xlabel("Predicted parity")
plt.ylabel("True parity")
plt.xticks([0, 1], ["0", "1"])
plt.yticks([0, 1], ["0", "1"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar()
plt.tight_layout()
exp.save_fig("confusion_matrix")
plt.show()

cm_df

## 9. Generate markdown summary

This writes:

```text
docs/02_topology_drift_demo.md
```

The markdown file points to repo-local figures and summarizes the key metrics.

In [ ]:
exp.write_md(
    title="Topology Drift Demo",
    metrics_dict={
        "Train accuracy": train_accuracy,
        "Test accuracy": test_accuracy,
        "Train drift rate": train_drift_rate,
        "Test drift rate": test_drift_rate,
        "Generalization gap": train_accuracy - test_accuracy,
    },
    figure_names=[
        "accuracy_vs_drift",
        "drift_by_hamming_weight",
        "residual_distribution",
        "confusion_matrix",
    ],
    interpretation="""
accuracy ≠ coherence
local fit → topology drift
residuals expose structural violation
""",
)

## 10. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
02_topology_drift_demo_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "02_topology_drift_demo_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 11. Takeaway

This notebook supports the second repo claim:

```text
models can fit locally while drifting from global structure
accuracy ≠ coherence
topology drift appears as residual structure
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → phase-lock stabilizes coherence against drift
```

Suggested next notebook:

```text
03_phase_lock_correction_loop.ipynb
```